# SynapseGPT Outlier Probe

Measures activation, gradient, and weight outliers in your **trained** SynapseGPT model.
Loads the latest checkpoint from Google Drive, runs forward+backward on a small synthetic batch,
and reports per-layer outlier statistics.

**Outlier definition**: For any tensor $x$:
- `rms = sqrt(mean(x^2))`
- `max_abs = max(abs(x))`
- `max_over_rms = max_abs / rms` (primary ranking metric)

Also reports: `mean_abs`, `p99_abs`, `p999_abs`, `p99_over_rms`, `p999_over_rms`.

**Gradients** are captured *before* gradient clipping.

All results auto-download as CSV + JSON when the notebook finishes.

In [ ]:
# @title 1. Clone repo, install deps
import os, sys, math, json, glob, gc, csv
from collections import defaultdict
from datetime import datetime
import numpy as np

REPO_URL = "https://github.com/ajencinas/synapse.git"
if not os.path.isdir("synapse"):
    !git clone {REPO_URL}
else:
    print("Already cloned.")

# Add tests/ to path so we can import synapse_model (not train.py!)
sys.path.insert(0, "./synapse/tests")

import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"  GPU: {props.name} | VRAM: {props.total_memory / 1024**3:.1f} GiB")

def vram(label=""):
    if device.type == "cuda":
        used = torch.cuda.memory_allocated() / 1024**3
        peak = torch.cuda.max_memory_allocated() / 1024**3
        print(f"  VRAM [{label}]: {used:.2f} GiB used, {peak:.2f} GiB peak")
    else:
        print(f"  VRAM [{label}]: N/A (CPU)")

In [ ]:
# @title 2. Mount Google Drive and find latest checkpoint
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

SYNAPSE_DIR = "/content/drive/MyDrive/synapse"
CHECKPOINT_DIR = os.path.join(SYNAPSE_DIR, "checkpoints")
EXPORT_DIR = "/content/synapse_outlier_results"
os.makedirs(EXPORT_DIR, exist_ok=True)

if not os.path.isdir(CHECKPOINT_DIR):
    raise FileNotFoundError(f"Checkpoint dir not found: {CHECKPOINT_DIR}")

ckpt_files = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, "*.pth")),
                    key=os.path.getmtime, reverse=True)

if not ckpt_files:
    raise FileNotFoundError(f"No .pth files found in {CHECKPOINT_DIR}")

print(f"Found {len(ckpt_files)} checkpoint(s). Latest 5:")
for i, p in enumerate(ckpt_files[:5]):
    size_gb = os.path.getsize(p) / 1024**3
    mtime = datetime.fromtimestamp(os.path.getmtime(p)).strftime("%Y-%m-%d %H:%M")
    print(f"  [{i}] {os.path.basename(p)}  ({size_gb:.2f} GB, {mtime})")

CHECKPOINT_PATH = ckpt_files[0]
print(f"\nSelected: {os.path.basename(CHECKPOINT_PATH)}")

In [ ]:
# @title 3. Read vocab size and show checkpoint metadata
META_PATH = os.path.join(SYNAPSE_DIR, "token_shards_merged", "meta.json")
if os.path.exists(META_PATH):
    with open(META_PATH) as f:
        meta = json.load(f)
    VOCAB_SIZE_RAW = int(meta["vocab_size"])
    VOCAB_SIZE = VOCAB_SIZE_RAW
    if VOCAB_SIZE % 64 != 0:
        VOCAB_SIZE = ((VOCAB_SIZE + 63) // 64) * 64
    print(f"Vocab: raw={VOCAB_SIZE_RAW}  padded={VOCAB_SIZE}")
else:
    VOCAB_SIZE = 131072
    print(f"meta.json not found, defaulting vocab_size={VOCAB_SIZE}")

# Inspect checkpoint metadata without loading weights
print("\nCheckpoint metadata:")
ckpt_obj = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
if isinstance(ckpt_obj, dict) and ckpt_obj.get("schema") == "v2":
    print(f"  Format:      v2")
    print(f"  Step:        {ckpt_obj.get('curr_step', '?')}")
    eval_hist = ckpt_obj.get("eval_history", [])
    if eval_hist:
        print(f"  Eval points: {len(eval_hist)}")
        last = eval_hist[-1]
        print(f"  Last eval:   step={last.get('step','?')}, loss={last.get('loss','?'):.4f}")
        by_src = last.get("by_source", {})
        if by_src:
            print(f"  Per-source losses:")
            for src, val in sorted(by_src.items()):
                print(f"    {src}: {val:.4f}")
    last_eval = ckpt_obj.get("last_eval_loss")
    if last_eval is not None:
        print(f"  Final eval:  {last_eval:.4f}")
    seen = ckpt_obj.get("seen_shards", [])
    print(f"  Seen shards: {len(seen)} (unique={len(set(seen))})")
elif isinstance(ckpt_obj, dict) and "model" in ckpt_obj:
    print("  Format: dict with 'model' key (no metadata)")
else:
    print("  Format: legacy (plain state_dict, no metadata)")

del ckpt_obj
gc.collect()

In [ ]:
# @title 4. Build model and load trained weights

from synapse_model import (
    SynapseGPT, TransformerBlock, CausalGQA, SwiGLU, RMSNorm,
    precompute_rope, apply_rope,
    EMBED_DIM, NUM_LAYERS, NUM_HEADS, NUM_KV_HEADS,
    FF_HIDDEN_DIM, RMSNORM_EPS, BLOCK_SIZE, ROPE_BASE,
)

model = SynapseGPT(VOCAB_SIZE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model built: {n_params/1e9:.3f}B params")
print(f"  embed={EMBED_DIM}, layers={NUM_LAYERS}, heads={NUM_HEADS}, kv_heads={NUM_KV_HEADS}")

# Load checkpoint weights
ckpt_obj = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
if isinstance(ckpt_obj, dict) and ckpt_obj.get("schema") == "v2":
    state_dict = ckpt_obj["model"]
elif isinstance(ckpt_obj, dict) and "model" in ckpt_obj:
    state_dict = ckpt_obj["model"]
else:
    state_dict = ckpt_obj

new_state = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
model_state = model.state_dict()
safe_state = {
    k: v for k, v in new_state.items()
    if k in model_state and v.shape == model_state[k].shape
}
model.load_state_dict(safe_state, strict=False)
print(f"Loaded {len(safe_state)}/{len(model_state)} layers")

del ckpt_obj, state_dict, new_state, model_state, safe_state
gc.collect()

model = model.to(device)
model.train()
vram("after model load")

print("\nModel architecture:")
for i, block in enumerate(model.blocks):
    attn = block.attn
    ff = block.ff
    print(f"  Block {i:>2}: Attn(q={attn.q_proj.weight.shape}, "
          f"k={attn.k_proj.weight.shape}, v={attn.v_proj.weight.shape}, "
          f"o={attn.o_proj.weight.shape}) | "
          f"SwiGLU(w1={ff.w1.weight.shape}, w2={ff.w2.weight.shape})")

In [ ]:
# @title 5. Outlier measurement utilities

def tensor_outlier_stats(x: torch.Tensor, name: str = "") -> dict:
    flat = x.detach().float().view(-1)
    if flat.numel() == 0:
        return {"name": name, "numel": 0}
    rms = torch.sqrt(flat.pow(2).mean()).item()
    max_abs = flat.abs().max().item()
    mean_abs = flat.abs().mean().item()
    sorted_abs = flat.abs().sort().values
    n = sorted_abs.numel()
    p99_abs = sorted_abs[int(0.99 * n)].item()
    p999_abs = sorted_abs[min(int(0.999 * n), n - 1)].item()
    max_over_rms = max_abs / rms if rms > 1e-12 else float("inf")
    p99_over_rms = p99_abs / rms if rms > 1e-12 else float("inf")
    p999_over_rms = p999_abs / rms if rms > 1e-12 else float("inf")
    return {
        "name": name, "numel": flat.numel(),
        "rms": round(rms, 6), "max_abs": round(max_abs, 6),
        "mean_abs": round(mean_abs, 6),
        "p99_abs": round(p99_abs, 6), "p999_abs": round(p999_abs, 6),
        "max_over_rms": round(max_over_rms, 4),
        "p99_over_rms": round(p99_over_rms, 4),
        "p999_over_rms": round(p999_over_rms, 4),
    }


def outlier_table(stats_list, top_k=25):
    valid = [s for s in stats_list if s.get("numel", 0) > 0]
    valid.sort(key=lambda s: s["max_over_rms"], reverse=True)
    return valid[:top_k]


def print_outlier_table(stats_list, title="Outlier Table"):
    print(f"\n{'=' * 130}")
    print(f"  {title}")
    print(f"{'=' * 130}")
    hdr = (f"{'#':>3} | {'Name':<55} | {'RMS':>9} | {'Max/RMS':>8} | "
            f"{'MaxAbs':>9} | {'P99/RMS':>8} | {'P999/RMS':>8} | {'MeanAbs':>9} | {'Numel':>8}")
    print(hdr)
    print("-" * 130)
    for i, s in enumerate(stats_list):
        line = (f"{i+1:>3} | {s['name']:<55} | {s['rms']:>9.4f} | {s['max_over_rms']:>8.2f} | "
                f"{s['max_abs']:>9.4f} | {s['p99_over_rms']:>8.2f} | {s['p999_over_rms']:>8.2f} | "
                f"{s['mean_abs']:>9.4f} | {s['numel']:>8}")
        print(line)
    print()

In [ ]:
# @title 6. Register forward hooks on ALL blocks

activation_store = {}

def make_hook(name):
    def hook(module, inp, out):
        if isinstance(out, (tuple, list)):
            out = out[0]
        activation_store[name] = out.detach()
    return hook

hooks = []

# Hook every block's main output
for i, block in enumerate(model.blocks):
    hooks.append(block.register_forward_hook(make_hook(f"block_{i:02d}.output")))

# Hook ALL blocks' internals: norm1, attn, attn.o_proj, norm2, ff
for i, block in enumerate(model.blocks):
    hooks.append(block.norm1.register_forward_hook(make_hook(f"block_{i:02d}.norm1")))
    hooks.append(block.attn.register_forward_hook(make_hook(f"block_{i:02d}.attn")))
    hooks.append(block.attn.o_proj.register_forward_hook(make_hook(f"block_{i:02d}.attn.o_proj")))
    hooks.append(block.norm2.register_forward_hook(make_hook(f"block_{i:02d}.norm2")))
    hooks.append(block.ff.register_forward_hook(make_hook(f"block_{i:02d}.ff")))

# Final norm + lm_head
hooks.append(model.final_norm.register_forward_hook(make_hook("final_norm")))
hooks.append(model.lm_head.register_forward_hook(make_hook("lm_head")))

print(f"Hooked {len(hooks)} points: {len(model.blocks)} blocks x 5 internals + 2 final")

In [ ]:
# @title 7. Run N forward passes and average activation stats
# Running multiple passes gives more stable activation outlier measurements
# because different random inputs may or may not trigger rare neurons.

N_PASSES = 3
B = 1
T = min(512, BLOCK_SIZE)

accumulated_act_stats = defaultdict(list)

print(f"Running {N_PASSES} forward passes (batch={B}, seq={T})...")
for pass_idx in range(N_PASSES):
    torch.manual_seed(42 + pass_idx)
    xb = torch.randint(0, VOCAB_SIZE, (B, T), device=device)
    yb = torch.randint(0, VOCAB_SIZE, (B, T), device=device)

    # Clear previous activations
    activation_store.clear()

    with torch.amp.autocast("cuda" if device.type == "cuda" else "cpu", dtype=torch.bfloat16):
        logits = model(xb)
        loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), yb.view(-1))

    print(f"  Pass {pass_idx+1}: loss={loss.item():.4f}")

    # Collect stats for this pass
    for name, act in activation_store.items():
        accumulated_act_stats[name].append(tensor_outlier_stats(act, name))

    # Free memory
    del logits, loss, xb, yb
    torch.cuda.empty_cache() if device.type == "cuda" else None

# Average across passes: RMS, mean_abs are averaged; max/rms computed from averaged values
activation_stats = []
for name, pass_stats in accumulated_act_stats.items():
    n = len(pass_stats)
    avg = {
        "name": name,
        "numel": pass_stats[0]["numel"],
        "rms": round(sum(s["rms"] for s in pass_stats) / n, 6),
        "max_abs": round(max(s["max_abs"] for s in pass_stats), 6),
        "mean_abs": round(sum(s["mean_abs"] for s in pass_stats) / n, 6),
        "p99_abs": round(sum(s["p99_abs"] for s in pass_stats) / n, 6),
        "p999_abs": round(sum(s["p999_abs"] for s in pass_stats) / n, 6),
    }
    avg["max_over_rms"] = round(avg["max_abs"] / avg["rms"], 4) if avg["rms"] > 1e-12 else float("inf")
    avg["p99_over_rms"] = round(avg["p99_abs"] / avg["rms"], 4) if avg["rms"] > 1e-12 else float("inf")
    avg["p999_over_rms"] = round(avg["p999_abs"] / avg["rms"], 4) if avg["rms"] > 1e-12 else float("inf")
    activation_stats.append(avg)

print(f"\nCollected activation stats from {N_PASSES} passes.")
vram("after activations")

In [ ]:
# @title 8. Run backward pass for gradients (BEFORE any clipping)
# Use the LAST forward pass's loss so gradients flow through trained weights

activation_store.clear()
torch.manual_seed(42)
xb = torch.randint(0, VOCAB_SIZE, (B, T), device=device)
yb = torch.randint(0, VOCAB_SIZE, (B, T), device=device)

with torch.amp.autocast("cuda" if device.type == "cuda" else "cpu", dtype=torch.bfloat16):
    logits = model(xb)
    loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), yb.view(-1))

loss.backward()
print(f"loss={loss.item():.4f} | backward complete (before any grad clipping)")

# Gradient stats
grad_stats = []
grad_count = 0
for name, param in model.named_parameters():
    if param.grad is None:
        continue
    grad_stats.append(tensor_outlier_stats(param.grad, name))
    grad_count += 1

print(f"Collected gradient stats for {grad_count} parameters.")
vram("after backward")

In [ ]:
# @title 9. Weight outlier stats

weight_stats = []
for name, param in model.named_parameters():
    weight_stats.append(tensor_outlier_stats(param.data, name))

print(f"Collected weight stats for {len(weight_stats)} parameters.")
vram("after weight stats")

In [ ]:
# @title 10. Print top outliers for all three categories

top_act = outlier_table(activation_stats, top_k=25)
top_grad = outlier_table(grad_stats, top_k=25)
top_w = outlier_table(weight_stats, top_k=25)

print_outlier_table(top_act, "Top Activation Outliers (by Max/RMS)")
print_outlier_table(top_grad, "Top Gradient Outliers (by Max/RMS)")
print_outlier_table(top_w, "Top Weight Outliers (by Max/RMS)")

In [ ]:
# @title 11. Bar plots of outlier ratios (split by scale)
try:
    import matplotlib.pyplot as plt

    def plot_outlier_bars(stats_list, title, top_k=20, ax=None, xlim=None):
        top = stats_list[:top_k][::-1]
        names = [s["name"][:50] for s in top]
        values = [s["max_over_rms"] for s in top]
        if ax is None:
            fig, ax = plt.subplots(figsize=(10, max(4, len(names) * 0.35)))
        colors = ["crimson" if v > 20 else "darkorange" if v > 10 else "steelblue" for v in values]
        ax.barh(names, values, color=colors)
        ax.set_xlabel("Max / RMS (outlier ratio)")
        ax.set_title(title)
        if xlim:
            ax.set_xlim(0, xlim)
        ax.axvline(x=10, color="darkorange", linestyle="--", alpha=0.5, label="mild (>10)")
        ax.axvline(x=20, color="crimson", linestyle="--", alpha=0.5, label="severe (>20)")
        ax.legend(loc="lower right", fontsize=8)
        ax.grid(axis="x", alpha=0.3)
        return ax

    # Figure 1: Activations + Gradients (large scale, similar range)
    fig1, axes1 = plt.subplots(1, 2, figsize=(22, 10))
    plot_outlier_bars(top_act, "Activation Outliers", top_k=20, ax=axes1[0])
    plot_outlier_bars(top_grad, "Gradient Outliers", top_k=20, ax=axes1[1])
    plt.tight_layout()
    plt.savefig(os.path.join(EXPORT_DIR, "outlier_barplots_act_grad.png"), dpi=150, bbox_inches="tight")
    plt.show()

    # Figure 2: Weights (much smaller scale, separate figure)
    fig2, ax2 = plt.subplots(1, 1, figsize=(14, max(4, 20 * 0.35)))
    plot_outlier_bars(top_w, "Weight Outliers", top_k=20, ax=ax2)
    plt.tight_layout()
    plt.savefig(os.path.join(EXPORT_DIR, "outlier_barplots_weights.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {EXPORT_DIR}/outlier_barplots_act_grad.png + outlier_barplots_weights.png")

except ImportError:
    print("matplotlib not available, skipping plots.")

In [ ]:
# @title 12. Per-layer trend plot (split by scale)
try:
    import matplotlib.pyplot as plt

    def extract_block_trends(stats_list, num_layers):
        block_act = []
        block_grad = []
        block_w = []
        for blk in range(num_layers):
            def avg_for(prefix, category):
                if category == "act":
                    vals = [s["max_over_rms"] for s in activation_stats if s["name"].startswith(f"block_{blk:02d}")]
                elif category == "grad":
                    vals = [s["max_over_rms"] for s in grad_stats if f"blocks.{blk}." in s["name"]]
                else:
                    vals = [s["max_over_rms"] for s in weight_stats if f"blocks.{blk}." in s["name"]]
                return max(vals) if vals else 0
            block_act.append(avg_for(None, "act"))
            block_grad.append(avg_for(None, "grad"))
            block_w.append(avg_for(None, "wt"))
        return block_act, block_grad, block_w

    act_trend, grad_trend, wt_trend = extract_block_trends(activation_stats, NUM_LAYERS)
    layers = list(range(NUM_LAYERS))

    # Figure 1: Activations + Gradients (large scale)
    fig1, ax1 = plt.subplots(figsize=(14, 5))
    ax1.plot(layers, act_trend, "o-", label="Activations", color="steelblue")
    ax1.plot(layers, grad_trend, "s-", label="Gradients", color="crimson")
    ax1.axhline(y=10, color="darkorange", linestyle="--", alpha=0.4)
    ax1.axhline(y=20, color="crimson", linestyle="--", alpha=0.4)
    ax1.set_xlabel("Block Index")
    ax1.set_ylabel("Max / RMS")
    ax1.set_title("Activation & Gradient Outlier Ratio Across Layers")
    ax1.set_xticks(layers)
    ax1.legend()
    ax1.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(EXPORT_DIR, "outlier_trend_act_grad.png"), dpi=150, bbox_inches="tight")
    plt.show()

    # Figure 2: Weights (small scale, separate figure)
    fig2, ax2 = plt.subplots(figsize=(14, 5))
    ax2.plot(layers, wt_trend, "^-", label="Weights", color="green")
    ax2.axhline(y=10, color="darkorange", linestyle="--", alpha=0.4)
    ax2.axhline(y=20, color="crimson", linestyle="--", alpha=0.4)
    ax2.set_xlabel("Block Index")
    ax2.set_ylabel("Max / RMS")
    ax2.set_title("Weight Outlier Ratio Across Layers")
    ax2.set_xticks(layers)
    ax2.legend()
    ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(EXPORT_DIR, "outlier_trend_weights.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {EXPORT_DIR}/outlier_trend_act_grad.png + outlier_trend_weights.png")

except ImportError:
    print("matplotlib not available, skipping plots.")

In [ ]:
# @title 13. Per-layer summary

print("=" * 80)
print("  Per-Block Summary: Max Max/RMS across sub-modules in each block")
print("=" * 80)
for blk in range(NUM_LAYERS):
    def blk_max(prefix, cat):
        if cat == "act":
            vals = [s["max_over_rms"] for s in activation_stats if s["name"].startswith(f"block_{blk:02d}")]
        elif cat == "grad":
            vals = [s["max_over_rms"] for s in grad_stats if f"blocks.{blk}." in s["name"]]
        else:
            vals = [s["max_over_rms"] for s in weight_stats if f"blocks.{blk}." in s["name"]]
        return f"{max(vals):.2f}" if vals else "---"
    print(f"  Block {blk:>2}: act={blk_max(None,'act'):>8}  |  grad={blk_max(None,'grad'):>8}  |  wt={blk_max(None,'wt'):>8}")
print()

## 18. Position-resolved outlier analysis (tests the early-token hypothesis)

**Hypothesis:** the large residual-stream `Max/RMS` ratios are concentrated at the
**first few token positions**, which have little/no causal context and so lean
harder on their own residual + MLP pathway (the attention-sink effect).

This cell:
1. Runs a forward pass with **real tokens** from a training shard *and* with
   **random tokens** (the data stream has no per-sequence BOS, so position 0 is
   simply the first token of the context window — the faithful test).
2. For every block output (residual stream) and FF output, computes the outlier
   ratio **per token position**, and locates the `(position, channel)` of the
   single global maximum.
3. Checks whether the spike is (a) anchored to early positions and (b) living in
   a **fixed hidden channel** across layers — the two signatures of a genuine
   massive-activation / attention-sink, as opposed to a measurement artifact.

If the global max sits at position 0–2 and in a consistent channel, your
explanation holds. If it smears across positions, it does not.

In [ ]:
# @title 18a. Capture full activations for real + random input
# Captures the FULL [B, T, H] tensors (not just summary stats) so we can slice
# the outlier ratio by token position and by hidden channel.

B_PROBE = 2
T_PROBE = min(512, BLOCK_SIZE)

# --- hooks that keep the full tensor (float32, on CPU to save VRAM) ---
full_store = {}
def make_full_hook(name):
    def hook(module, inp, out):
        if isinstance(out, (tuple, list)):
            out = out[0]
        full_store[name] = out.detach().float().cpu()   # [B, T, H]
    return hook

probe_hooks = []
for i, block in enumerate(model.blocks):
    probe_hooks.append(block.register_forward_hook(make_full_hook(f"block_{i:02d}.output")))
    probe_hooks.append(block.ff.register_forward_hook(make_full_hook(f"block_{i:02d}.ff")))

def run_capture(idx_tensor):
    full_store.clear()
    with torch.no_grad(), torch.amp.autocast(
        "cuda" if device.type == "cuda" else "cpu", dtype=torch.bfloat16):
        model(idx_tensor)
    # also report non-finite counts (explicit NaN/Inf check)
    nonfinite = {k: int((~torch.isfinite(v)).sum()) for k, v in full_store.items()}
    bad = {k: n for k, n in nonfinite.items() if n > 0}
    if bad:
        print(f"  !! non-finite values detected: {bad}")
    else:
        print(f"  all {len(full_store)} captured tensors are finite (no NaN/Inf)")
    return {k: v.clone() for k, v in full_store.items()}

# --- build the two inputs ---
captures = {}

# random tokens
torch.manual_seed(123)
rand_idx = torch.randint(0, VOCAB_SIZE, (B_PROBE, T_PROBE), device=device)
print("Forward pass: RANDOM tokens")
captures["random"] = run_capture(rand_idx)

# real tokens from a training shard (faithful, in-distribution input)
try:
    shard_dir = os.path.join(SYNAPSE_DIR, "token_shards_merged")
    with open(os.path.join(shard_dir, "meta.json")) as f:
        shard_dtype = np.dtype(json.load(f).get("shard_dtype", "uint16"))
    shards = sorted(glob.glob(os.path.join(shard_dir, "shard_*.bin")))
    if not shards:
        raise FileNotFoundError("no shard_*.bin found")
    need = B_PROBE * T_PROBE
    toks = np.fromfile(shards[0], dtype=shard_dtype, count=need).astype(np.int64)
    if toks.size < need:
        raise ValueError(f"shard too small: got {toks.size}, need {need}")
    real_idx = torch.from_numpy(toks).clamp_(0, VOCAB_SIZE - 1).view(B_PROBE, T_PROBE).to(device)
    print(f"Forward pass: REAL tokens from {os.path.basename(shards[0])} (dtype={shard_dtype})")
    captures["real"] = run_capture(real_idx)
except Exception as e:
    print(f"Real-token capture skipped ({e}); analysing random only")

for h in probe_hooks:
    h.remove()
print(f"\nCaptured {len(captures)} input type(s): {list(captures)}")
vram("after position capture")

In [ ]:
# @title 18b. Per-position & per-channel breakdown + verdict

def position_breakdown(x):
    # x: [B, T, H] float
    B, T, H = x.shape
    abs_x = x.abs()
    global_rms = x.pow(2).mean().sqrt().item()
    per_pos_max = abs_x.amax(dim=(0, 2))                 # [T]
    per_pos_rms = x.pow(2).mean(dim=(0, 2)).sqrt()       # [T]
    ratio_global = (per_pos_max / (global_rms + 1e-8))   # comparable across positions
    ratio_local = (per_pos_max / (per_pos_rms + 1e-8))   # outlierness within a position
    # locate the single global maximum
    flat = abs_x.view(-1)
    gi = int(flat.argmax())
    b = gi // (T * H); t = (gi // H) % T; h = gi % H
    # which channel carries the most energy overall
    chan_energy = x.pow(2).mean(dim=(0, 1))             # [H]
    massive_chan = int(chan_energy.argmax())
    chan_frac = (chan_energy.max() / chan_energy.sum()).item()
    return {
        "T": T, "H": H, "global_rms": global_rms,
        "ratio_global": ratio_global, "ratio_local": ratio_local,
        "argmax_pos": t, "argmax_chan": h, "argmax_val": flat[gi].item(),
        "massive_chan": massive_chan, "chan_frac": chan_frac,
    }

EARLY, MID = 8, 128
for inp_name, cap in captures.items():
    print("=" * 100)
    print(f"  INPUT = {inp_name.upper()}   (residual-stream block outputs)")
    print("=" * 100)
    print(f"{'blk':>3} | {'argmax@(pos,chan)':>18} | {'maxAbs':>9} | "
          f"{'r@pos0':>7} | {'early':>7} | {'mid':>7} | {'late':>7} | {'massCh':>6} | {'chFrac':>6}")
    print("-" * 100)
    block_summ = []
    for blk in range(NUM_LAYERS):
        x = cap.get(f"block_{blk:02d}.output")
        if x is None:
            continue
        d = position_breakdown(x)
        rg = d["ratio_global"]
        early = rg[:EARLY].mean().item()
        mid = rg[EARLY:MID].mean().item()
        late = rg[MID:].mean().item()
        block_summ.append((blk, d, early, mid, late))
        print(f"{blk:>3} | {('(%d,%d)' % (d['argmax_pos'], d['argmax_chan'])):>18} | "
              f"{d['argmax_val']:>9.1f} | {rg[0].item():>7.1f} | {early:>7.1f} | "
              f"{mid:>7.1f} | {late:>7.1f} | {d['massive_chan']:>6} | {d['chan_frac']:>6.2f}")

    # ---- verdict for this input ----
    pos = [d["argmax_pos"] for _, d, *_ in block_summ]
    chans = [d["massive_chan"] for _, d, *_ in block_summ]
    early_frac = sum(p < EARLY for p in pos) / len(pos)
    from collections import Counter
    top_chan, top_n = Counter(chans).most_common(1)[0]
    e_mean = np.mean([e for *_, e, m, l in block_summ])
    l_mean = np.mean([l for *_, e, m, l in block_summ])
    print("-" * 100)
    print(f"  global-max at position <{EARLY}: {early_frac*100:.0f}% of blocks  | "
          f"mean ratio  early={e_mean:.0f}  late={l_mean:.0f}  (early/late={e_mean/max(l_mean,1e-9):.2f}x)")
    print(f"  most common 'massive' channel: #{top_chan} in {top_n}/{len(chans)} blocks")
    verdict = ("SUPPORTED: outliers concentrate at early positions"
               if early_frac >= 0.5 and e_mean > 1.3 * l_mean
               else "NOT clearly early-position driven: spike is spread across positions")
    print(f"  --> hypothesis {verdict}")
    print(f"  --> fixed-channel sink: {'YES' if top_n >= 0.5*len(chans) else 'NO (channel varies)'}")
    print()

In [ ]:
# @title 18c. Plots: outlier ratio vs token position
try:
    import matplotlib.pyplot as plt

    SHOW_BLOCKS = [b for b in [3, 4, 5, 9, 13, 19, NUM_LAYERS - 1] if b < NUM_LAYERS]
    EXPORT_DIR = globals().get("EXPORT_DIR", ".")

    for inp_name, cap in captures.items():
        fig, axes = plt.subplots(1, 2, figsize=(20, 6))
        for blk in SHOW_BLOCKS:
            x = cap.get(f"block_{blk:02d}.output")
            if x is None:
                continue
            d = position_breakdown(x)
            rg = d["ratio_global"].numpy()
            axes[0].plot(rg, label=f"block {blk}")           # full sequence
            axes[1].plot(rg[:32], marker="o", ms=3, label=f"block {blk}")  # zoom early
        axes[0].set_title(f"[{inp_name}] residual Max/RMS vs position (full sequence)")
        axes[1].set_title(f"[{inp_name}] residual Max/RMS vs position (first 32 tokens)")
        for ax in axes:
            ax.set_xlabel("token position"); ax.set_ylabel("max_abs / global_rms")
            ax.grid(alpha=0.3); ax.legend(fontsize=8, ncol=2)
        plt.tight_layout()
        out = os.path.join(EXPORT_DIR, f"outlier_by_position_{inp_name}.png")
        plt.savefig(out, dpi=140, bbox_inches="tight"); plt.show()
        print(f"Saved {out}")

    # heatmap: position (x) vs block (y) for the residual stream, real input if present
    inp_name = "real" if "real" in captures else "random"
    cap = captures[inp_name]
    T = position_breakdown(cap["block_00.output"])["T"]
    mat = np.zeros((NUM_LAYERS, min(T, 64)))
    for blk in range(NUM_LAYERS):
        x = cap.get(f"block_{blk:02d}.output")
        if x is None:
            continue
        mat[blk] = position_breakdown(x)["ratio_global"][:mat.shape[1]].numpy()
    fig, ax = plt.subplots(figsize=(16, 7))
    im = ax.imshow(mat, aspect="auto", cmap="magma", origin="lower")
    ax.set_xlabel("token position (first 64)"); ax.set_ylabel("block index")
    ax.set_title(f"[{inp_name}] residual Max/RMS — position x block (bright = outlier)")
    fig.colorbar(im, ax=ax, label="max_abs / global_rms")
    plt.tight_layout()
    out = os.path.join(EXPORT_DIR, f"outlier_heatmap_{inp_name}.png")
    plt.savefig(out, dpi=140, bbox_inches="tight"); plt.show()
    print(f"Saved {out}")

except ImportError:
    print("matplotlib not available, skipping plots.")

# cleanup big tensors
del captures, full_store
gc.collect()
torch.cuda.empty_cache() if device.type == "cuda" else None

In [ ]:
# @title 14. Export all results to CSV and JSON


timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = os.path.join(EXPORT_DIR, f"outlier_results_{timestamp}.csv")
json_path = os.path.join(EXPORT_DIR, f"outlier_results_{timestamp}.json")

# CSV: flat table with all three categories
csv_rows = []
for cat, stats in [("activation", activation_stats),
                   ("gradient", grad_stats),
                   ("weight", weight_stats)]:
    for s in stats:
        row = {"category": cat, **s}
        csv_rows.append(row)

csv_rows.sort(key=lambda r: r["max_over_rms"], reverse=True)

with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=csv_rows[0].keys())
    writer.writeheader()
    writer.writerows(csv_rows)

# JSON: structured by category
json_data = {
    "checkpoint": os.path.basename(CHECKPOINT_PATH),
    "model_params": n_params,
    "vocab_size": VOCAB_SIZE,
    "block_size": BLOCK_SIZE,
    "num_layers": NUM_LAYERS,
    "embed_dim": EMBED_DIM,
    "n_activation_stats": len(activation_stats),
    "n_gradient_stats": len(grad_stats),
    "n_weight_stats": len(weight_stats),
    "activation": activation_stats,
    "gradient": grad_stats,
    "weight": weight_stats,
    "top_activation": top_act,
    "top_gradient": top_grad,
    "top_weight": top_w,
}

with open(json_path, "w") as f:
    json.dump(json_data, f, indent=2)

print(f"Exported:")
print(f"  CSV:  {csv_path}")
print(f"  JSON: {json_path}")
print(f"  Rows: {len(csv_rows)} total")
print(f"  Size: {os.path.getsize(csv_path)/1024:.1f} KB (csv), {os.path.getsize(json_path)/1024:.1f} KB (json)")

In [ ]:
# @title 15. Auto-download all results from Colab
from google.colab import files

print("Downloading results to your machine...")
for fname in os.listdir(EXPORT_DIR):
    fpath = os.path.join(EXPORT_DIR, fname)
    if os.path.isfile(fpath):
        print(f"  -> {fname}")
        files.download(fpath)

print("\nAll files downloaded.")

In [ ]:
# @title 16. Interpretation

print("""
========================= INTERPRETATION =========================

Metric definitions
------------------
  RMS       = sqrt(mean(x^2))       -- typical scale of the tensor
  Max/RMS   = max_abs / rms          -- how many sigma the largest element is
  P99/RMS   = 99th percentile / rms  -- tail extremity (less noisy than max)
  P999/RMS  = 99.9th percentile / rms

What high values mean
---------------------
Max/RMS ~ 3-5     Normal. Typical for Gaussian-like distributions.
Max/RMS ~ 6-10    Mild outliers. A few activations/weights are noticeably
                  larger than the bulk distribution.
Max/RMS ~ 10-30   Moderate outliers. Common in early-training transformer
                  FFN layers before the model learns to self-stabilize.
Max/RMS > 30      Severe outliers. May indicate numerical instability or
                  a "magnet" neuron that dominates the update. Could lead
                  to loss spikes, especially if gradients are also high.

Gradient outliers deserve special attention:
  - High grad Max/RMS with low weight Max/RMS  ->  intermittent large updates
    that may destabilize training (watch for loss spikes).
  - High grad AND weight Max/RMS  ->  a persistent outlier feature that the
    model relies on heavily. May limit quantizability (QLoRA / SmoothQuant).

Activation outliers:
  - Concentrated in SwiGLU (ff) outputs: expected -- the gating mechanism can
    produce large positive values from SiLU(x) * linear projection.
  - In attention o_proj: may indicate that a few token positions dominate
    the attention-weighted sum (softmax not diffuse enough).
  - In RMSNorm outputs: RMSNorm explicitly rescales to RMS ~ 1, so
    Max/RMS ~ max_abs (since rms ~ 1). Values > 10 here are noteworthy.

Implications for quantization (e.g. INT8 / FP8 training):
  - Layers with Max/RMS > 10 need per-tensor or per-channel scaling factors
    to avoid clipping the tail.
  - Layers with Max/RMS > 30 may require mixed-precision (keeping those
    activations in FP16/BF16) even in an otherwise quantized pipeline.
==================================================================
""")

In [ ]:
# @title 17. Cleanup
for h in hooks:
    h.remove()
del activation_store, accumulated_act_stats
gc.collect()
torch.cuda.empty_cache() if device.type == "cuda" else None
vram("final")
print(f"\nDone. {len(hooks)} hooks removed.")